# Day 2 - words (GPU session only for the ISL training cell)
Inputs: `slr-code`, your Day-1 output dataset, `209sontung/sign-language`, `asl-signs`.

In [ ]:
# Code bundle: upload dist/slr_code.zip as a Kaggle Dataset named "slr-code", then add it as input.
import glob, os, shutil, subprocess, sys
src = (glob.glob("/kaggle/input/slr-code*") or glob.glob("/content/slr_code"))[0]
shutil.copytree(src, "/kaggle/working/code" if os.path.exists("/kaggle") else "/content/code", dirs_exist_ok=True)
WORK = "/kaggle/working" if os.path.exists("/kaggle") else "/content"
CODE = f"{WORK}/code"
os.chdir(CODE)
# mediapipe 0.10.14 still has the `mp.solutions` API we use. If pip complains about numpy, run
# `!pip install "numpy<2"` and restart the session once.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mediapipe==0.10.14", "remotezip", "opencv-python-headless"])

## ASL: pretrained inference on your chosen 20-30 signs (no training)

In [ ]:
MODEL_DIR = "/kaggle/input/sign-language"
SIGNS = "hello thankyou please yes no mom dad water eat drink go sleep happy sad like want home cat dog bird red blue green hungry sick".split()
!python scripts/asl_words.py infer --model-dir $MODEL_DIR --data /kaggle/input/asl-signs --signs {" ".join(SIGNS)} --per-sign 20 --out $WORK/asl_pretrained_eval.json

### Optional: train/fine-tune only on those signs (also the fallback if the pretrained weights won't load)

In [ ]:
!python scripts/asl_words.py extract --data /kaggle/input/asl-signs --signs {" ".join(SIGNS)} --out $WORK/asl_words
!python scripts/train_words.py --data $WORK/asl_words --out $WORK/models/asl_words.pt --epochs 120

## ISL: train on the INCLUDE-50 keypoints from Day 1 (enable GPU T4)

In [ ]:
INCLUDE_KP = "/kaggle/input/REPLACE-WITH-DAY1-OUTPUT-DATASET/include50"   # or $WORK/include50 if same session
!python scripts/train_words.py --data $INCLUDE_KP --out $WORK/models/isl_words.pt --epochs 150 --seeds 3

## Package the small models for download (a few MB each)

In [ ]:
!cp /kaggle/input/REPLACE-WITH-DAY1-OUTPUT-DATASET/models/*.joblib $WORK/models/ 2>/dev/null
!cd $WORK && zip -r models.zip models && ls -lh models.zip